# Loading Libraries

In [1]:
import io, os
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv

In [2]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI 
from langgraph.graph import StateGraph, END, START, add_messages
from typing import TypedDict, Annotated, Optional
from pydantic import BaseModel

In [3]:
from langsmith import traceable

# Loading the Model

In [4]:
load_dotenv()

True

In [5]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# Playing with the deprecated model
# OPEN_AI_MODEL = os.getenv("OPEN_AI_MODEL_DEP")
# Playing with the frontier model
OPEN_AI_MODEL = os.getenv("OPEN_AI_MODEL")

In [43]:
OPEN_AI_MODEL

'gpt-4o-mini'

In [65]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "CodeReviewMAS_EXT"

In [47]:
open_ai_model = ChatOpenAI(model_name=OPEN_AI_MODEL, openai_api_key=OPENAI_API_KEY, temperature=0, max_tokens=1000)

# Develop the agents

In [48]:
class State(TypedDict):
    pr_title: str
    pr_description: str
    file_path: str
    changed_code: str
    correctness: str
    security: str
    performance: str
    combined_output: str

In [49]:
TRIAGE_SYSTEM_PROMPT="""You are the Triage Agent in a multi-agent code review system.

Your role is to orchestrate the review process, not to review the code yourself.

Given a pull request, perform the following steps:

1. Understand the purpose of the code changes by reading the PR title, description, and the changed code.
2. Divide the review into three independent tasks and assign them to the following specialized agents:
   - Correctness Reviewer: Verify logic, functionality, and API usage.
   - Security Reviewer: Identify security vulnerabilities and unsafe coding practices.
   - Performance Reviewer: Detect performance bottlenecks and inefficient implementations.
Do not perform the technical review yourself or invent findings. Your responsibility is limited to task delegation.
"""

In [50]:
CORRECTNESS_SYSTEM_PROMPT="""You are the Correctness Reviewer in a multi-agent code review system.

Review the given code changes only for functional correctness. Identify:
- Logic errors and incorrect behavior
- API misuse
- Missing edge cases
- Exception handling issues
- Bugs introduced by the changes

For each issue, provide:
- Severity (Critical/High/Medium/Low)
- Location (file/line if available)
- Description
- Suggested fix

Keep the review comments concise.
Do not review security, performance, or code style. If no issues are found, state: "No correctness issues found.
"""

In [51]:
SECURITY_SYSTEM_PROMPT="""You are the Security Reviewer in a multi-agent code review system.

Review the code changes only for security issues. Identify:
- Vulnerabilities and unsafe practices
- Authentication and authorization flaws
- Input validation issues
- Injection risks
- Sensitive data exposure
- Insecure dependencies or configurations

For each issue, provide:
- Severity (Critical/High/Medium/Low)
- Location (file/line if available)
- Description
- Suggested fix

Keep the review comments concise.
Do not review correctness, performance, or code style. If no security issues are found, state: "No security issues found."""

In [52]:
PERFORMANCE_SYSTEM_PROMPT="""You are the Performance Reviewer in a multi-agent code review system.

Review the code changes only for performance issues. Identify:
- Inefficient algorithms or unnecessary computation
- Slow database, API, or I/O operations
- Memory inefficiencies or resource leaks
- Scalability bottlenecks
- Unnecessary duplication or repeated work

For each issue, provide:
- Severity (Critical/High/Medium/Low)
- Location (file/line if available)
- Description
- Suggested optimization

Keep the review comments concise.
Do not review correctness, security, or code style. If no performance issues are found, state: "No performance issues found.
"""

In [53]:
AGGREGATE_SYSTEM_PROMPT="""You are the Aggregator Agent in a multi-agent code review system.

Your role is to combine findings from specialized reviewers into a final review report.

Tasks:
- Collect and summarize findings from correctness, security, and performance reviewers.
- Remove duplicate findings.
- Highlight conflicting opinions.
- Prioritize issues by severity.
- Provide an overall recommendation:
  APPROVE, REQUEST CHANGES, or COMMENT ONLY.

Do not introduce new findings or perform your own code analysis. Base the report only on reviewer feedback.
"""

In [54]:
@traceable(
    run_type="chain",
    name="Triage agent"
)
def triage_code_review(state: State):
    """Coordinate the code review task and assign to appropriate reviewers."""
    response = open_ai_model.invoke([SystemMessage(content=TRIAGE_SYSTEM_PROMPT), 
                                     HumanMessage(content=f"PR title: {state['pr_title']}\nPR description: {state['pr_description']}\nChanged code: {state['changed_code']}")
    ])
    return state

In [55]:
@traceable(
    run_type="chain",
    name="Correctness Reviewer"
)
def review_correctness(state: State):
    response = open_ai_model.invoke([SystemMessage(content=CORRECTNESS_SYSTEM_PROMPT), 
                                     HumanMessage(content=f"Changed code: {state['changed_code']}")])
    return {"correctness": response.content}

In [56]:
@traceable(
    run_type="chain",
    name="Security Reviewer"
)
def review_security_issues(state: State):
    response = open_ai_model.invoke([SystemMessage(content=SECURITY_SYSTEM_PROMPT), 
                                     HumanMessage(content=f"Changed code: {state['changed_code']}")])
    return {"security": response.content}

In [57]:
@traceable(
    run_type="chain",
    name="Performance Reviewer"
)
def review_performance(state: State):
    response = open_ai_model.invoke([SystemMessage(content=PERFORMANCE_SYSTEM_PROMPT), 
                                     HumanMessage(content=f"Changed code: {state['changed_code']}")])
    return {"performance":response.content} 

In [58]:
@traceable(
    run_type="chain",
    name="Aggregator Agent"
)
def aggregate_findings(state: State):
    response = open_ai_model.invoke([SystemMessage(content=AGGREGATE_SYSTEM_PROMPT),        
                                     HumanMessage(content=f"Correctness findings: {state["correctness"]}\nSecurity findings: {state["security"]}\nPerformance findings: {state["performance"]}")])
    return {"combined_output": response.content}

# Develop the LangGraph

In [59]:
def devlop_compiled_network():
    code_review_network = StateGraph(State)
    code_review_network.add_node("triage", triage_code_review)
    code_review_network.add_node("correctness", review_correctness)
    code_review_network.add_node("security", review_security_issues)
    code_review_network.add_node("performance", review_performance)
    code_review_network.add_node("aggregate", aggregate_findings)

    code_review_network.add_edge(START, "triage")
    code_review_network.add_edge("triage", "correctness")
    code_review_network.add_edge("triage", "security")
    code_review_network.add_edge("triage", "performance")
    code_review_network.add_edge("correctness", "aggregate")
    code_review_network.add_edge("security", "aggregate")
    code_review_network.add_edge("performance", "aggregate")
    code_review_network.add_edge("aggregate", END)

    return code_review_network.compile()   

In [60]:
code_review_mas=devlop_compiled_network()

# Diagnosis & Testing

In [61]:
pr_title="Avoid accessing non-existing \"$ref\" key for Pydantic v2 compat remapping"
pr_description="cf Discussion #14265"
review_comment="For my case reported in https://github.com/fastapi/fastapi/discussions/14265 I am able to successfully generate an OpenAPI specification file without erroring out when using this branch of FastAPI.\r\n\r\nThanks!\nThanks for reporting back, @jerry-reevo !\nJust adding in, my team was also facing the same issue discussed in this PR which has kept us stuck at `fastapi==0.118.3`. This branch has seemed to resolved this issue for me (at least for local testing).\nI can also confirm this PR fixes the issue for me. Thanks @svlandeg!"
changed_code="old_name_to_new_name_map = {}\n    for field_key, schema in field_mapping.items():\n        model = field_key[0].type_\n        if model not in model_name_map:\n        if model not in model_name_map or \"$ref\" not in schema:\n            continue\n        new_name = model_name_map[model]\n        old_name = schema[\"$ref\"].split(\"/\")[-1]"

In [62]:
import langsmith as ls

# Review the Code

In [63]:
def load_repo_data(repo_path):
    with open(repo_path, "r") as f:
        return f.read()

In [64]:
def load_code_changes(json_content):
   return json.loads(json_content)

In [66]:
def review_code_from_pr(element, ls_config):
    if element["changed_code"] is not None:
        pr_title, pr_description=element["pr_title"], element["pr_description"]
        changed_code=element["changed_code"]
        with ls.tracing_context(enable=True, project_name="CodeReviewMAS_EXT", run_name="CodeReviewMAS_EXT", metadata=ls_config["metadata"]):
            review_comment = code_review_mas.invoke({"pr_title": pr_title, "pr_description": pr_description, "changed_code": changed_code}, config=ls_config)
        ground_truth = element["review_comments"]
        return {"pr_number": element["pr_number"], "repo": element["repo"], "file_path": element["file_path"], 
                "review_comment": review_comment["combined_output"], "ground_truth": ground_truth}

In [67]:
def make_ls_config(repo_name, first_element):
    ls_config = {
        "metadata": {
            "repository": repo_name,
            "pr_number": first_element["pr_number"],
            "repo": first_element["repo"],
            "file_path": first_element["file_path"]
        }
    }
    return ls_config

In [68]:
def review_and_save_from_pr(element, ls_config):
    evaluated_result=review_code_from_pr(element, ls_config)
    return evaluated_result

In [28]:
import time

In [69]:
def save_outputs(out_file, saved_outputs):
    with open(out_file, "w") as f:
        json.dump(saved_outputs, f)

# Run the Experiment

In [70]:
def run_experiment(repo_name, elements, out_file):
    outputs=[]
    for element in elements:
        ls_config = make_ls_config(repo_name, element)
        evaluated_result=review_and_save_from_pr(element, ls_config)
        outputs.append(evaluated_result)
        time.sleep(1)
        
    save_outputs(out_file, outputs)

In [71]:
def get_elements(repo_path):
    json_repo_data=load_repo_data(repo_path)
    elements=load_code_changes(json_repo_data)
    return elements

In [ ]:
repo_name="pandas"
repo_path="dataset/processed/pandas.json"
output_path="dataset/reviewed-ext/pandas.json"
elements=get_elements(repo_path)
run_experiment(repo_name, elements, output_path)

: 